# MA3632 — Workshop 10: Model Evaluation and Selection

This workshop accompanies Lecture 10. We examine the train/validate/test
protocol carefully, revisit cross-validation as an estimator rather than a
tuning tool, explore the multiple comparisons problem and nested CV, compare
two classifiers using McNemar's test, and study probability calibration via
reliability diagrams, the Brier score, and Platt scaling.

Work through all parts in order. Take-home exercises are at the end.

---

## Part A — The train / validate / test protocol

A single held-out test set gives one unbiased estimate of generalisation error
— but only if it is never used for any decision during model development.
Every time the test set influences a choice (of algorithm, hyperparameter, or
feature set) it leaks information and the final error estimate becomes
optimistic.  We demonstrate this with a deliberate data-snooping experiment.

### A1. Imports and data

In [ ]:
import matplotlib
matplotlib.use('Agg')
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, load_wine, make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split, cross_val_score, StratifiedKFold,
    LeaveOneOut, GridSearchCV, cross_validate
)
from sklearn.metrics import (
    accuracy_score, confusion_matrix, roc_auc_score,
    brier_score_loss, ConfusionMatrixDisplay
)
from sklearn.calibration import (
    CalibratedClassifierCV, CalibrationDisplay, calibration_curve
)
from sklearn.preprocessing import label_binarize
from scipy.stats import chi2, binom
import warnings
warnings.filterwarnings("ignore")

digits = load_digits()
X_dig, y_dig = digits.data, digits.target

wine = load_wine()
X_wine, y_wine = wine.data, wine.target

# Single train / test split for Parts A and D
X_tr, X_te, y_tr, y_te = train_test_split(
    X_dig, y_dig, test_size=0.25, random_state=0, stratify=y_dig
)
print(f"Digits  — train: {X_tr.shape}, test: {X_te.shape}")
print(f"Wine    — {X_wine.shape}, {len(np.unique(y_wine))} classes")

### A2. Data snooping: the cost of peeking at the test set

In [ ]:
# Simulate a researcher who repeatedly picks the best hyperparameter
# by evaluating on the test set rather than a validation set.
# We compare this to an honest evaluation on a separate held-out set.

rng = np.random.default_rng(1)
n_trials = 40
C_grid = np.logspace(-2, 2, 8)

# Split into train / snoop-test / honest-test
X_tr2, X_rest, y_tr2, y_rest = train_test_split(
    X_dig, y_dig, test_size=0.4, random_state=2, stratify=y_dig
)
X_snoop, X_honest, y_snoop, y_honest = train_test_split(
    X_rest, y_rest, test_size=0.5, random_state=2, stratify=y_rest
)

snooped_best = []
honest_best  = []

for _ in range(n_trials):
    # Shuffle labels on a small random noise dataset to simulate variance
    idx = rng.choice(len(X_tr2), size=len(X_tr2), replace=True)
    Xb, yb = X_tr2[idx], y_tr2[idx]

    # Snooper picks C that minimises snoop-test error
    accs_snoop = [
        accuracy_score(y_snoop,
            LogisticRegression(C=c, max_iter=300).fit(Xb, yb).predict(X_snoop))
        for c in C_grid
    ]
    best_c_snoop = C_grid[np.argmax(accs_snoop)]
    clf_s = LogisticRegression(C=best_c_snoop, max_iter=300).fit(Xb, yb)
    snooped_best.append(accuracy_score(y_snoop,  clf_s.predict(X_snoop)))
    honest_best.append( accuracy_score(y_honest, clf_s.predict(X_honest)))

print(f"Mean accuracy on snooped set  (optimistically biased): {np.mean(snooped_best):.4f}")
print(f"Mean accuracy on honest set   (true generalisation):   {np.mean(honest_best):.4f}")
print(f"Optimism bias: {np.mean(snooped_best) - np.mean(honest_best):+.4f}")

**Exercise A.** The snooped accuracy is consistently higher than the honest
accuracy.  Explain in your own words why this happens, and describe one
practical safeguard that prevents it in a real modelling workflow.

## Part B — Cross-validation as an estimator

$k$-fold CV estimates the expected error of the *fitting procedure* applied to
a dataset of size $n(1-1/k)$, not the error of the final model trained on all
$n$ points.  Here we study variance across folds, the effect of $k$, and the
importance of stratification for imbalanced classes.

### B1. Fold-by-fold variance

In [ ]:
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=3)
rf  = RandomForestClassifier(n_estimators=100, random_state=3)

fold_scores = []
for train_idx, val_idx in skf.split(X_dig, y_dig):
    rf.fit(X_dig[train_idx], y_dig[train_idx])
    fold_scores.append(accuracy_score(y_dig[val_idx], rf.predict(X_dig[val_idx])))

fold_scores = np.array(fold_scores)
print(f"10-fold CV scores: {fold_scores.round(4)}")
print(f"Mean: {fold_scores.mean():.4f}   Std: {fold_scores.std():.4f}")
print(f"95% interval (mean +/- 2*std/sqrt(k)): "
      f"({fold_scores.mean() - 2*fold_scores.std()/np.sqrt(10):.4f}, "
      f"{fold_scores.mean() + 2*fold_scores.std()/np.sqrt(10):.4f})")

fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar(range(1, 11), fold_scores, color='steelblue', alpha=0.8)
ax.axhline(fold_scores.mean(), color='tomato', lw=1.5, label=f'Mean={fold_scores.mean():.4f}')
ax.axhline(fold_scores.mean() - fold_scores.std(), color='grey', ls='--', lw=1)
ax.axhline(fold_scores.mean() + fold_scores.std(), color='grey', ls='--', lw=1,
           label='+/- 1 std')
ax.set_xlabel('Fold')
ax.set_ylabel('Accuracy')
ax.set_title('Fold-by-fold accuracy — Random Forest on Digits (10-fold CV)')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/b1_folds.png', dpi=110)
plt.close()
print("Saved b1_folds.png")

### B2. Effect of $k$ on bias and variance of the CV estimate

In [ ]:
k_values = [2, 3, 5, 10, 20, 50]
cv_means = []
cv_stds  = []

for k in k_values:
    skf_k = StratifiedKFold(n_splits=k, shuffle=True, random_state=4)
    scores = cross_val_score(
        RandomForestClassifier(n_estimators=100, random_state=4),
        X_dig, y_dig, cv=skf_k, scoring='accuracy'
    )
    cv_means.append(scores.mean())
    cv_stds.append(scores.std())

cv_means = np.array(cv_means)
cv_stds  = np.array(cv_stds)

fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(k_values, cv_means, yerr=cv_stds, fmt='o-',
            color='steelblue', capsize=4, label='Mean +/- 1 std across folds')
ax.set_xlabel('k (number of folds)')
ax.set_ylabel('CV accuracy')
ax.set_title('CV estimate vs k — Random Forest on Digits')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/b2_k_effect.png', dpi=110)
plt.close()
print(f"{'k':>4}  {'mean':>7}  {'std':>7}")
for k, m, s in zip(k_values, cv_means, cv_stds):
    print(f"{k:>4}  {m:.4f}  {s:.4f}")

### B3. Stratification matters for imbalanced data

When a classifier already performs near ceiling on a task, every fold's
score sits close to the maximum regardless of how folds are assigned, and
no comparison of fold-to-fold variance can reveal anything about
stratification. The comparison below therefore uses a deliberately
weaker classifier (a shallow decision tree) on the naturally imbalanced
digit-8-vs-rest problem (about 10% positive), and metrics sensitive to
the minority class (precision, F1) alongside accuracy, which is
dominated by the majority class and least likely to show the effect.

In [ ]:
from sklearn.model_selection import KFold

y_imb = (y_dig == 8).astype(int)
print(f"Class balance: {y_imb.mean():.3f} positive (digit 8)")

clf_weak = DecisionTreeClassifier(max_depth=3, random_state=0)

# A single stratified-vs-non-stratified comparison, at one fixed seed, is
# itself a noisy quantity (Section 2.3): the fold assignment is random, so
# the two variance estimates below could in principle come out in either
# order just from that seed's luck. We first look at one seed, then repeat
# over many seeds to see the expected direction reliably.
seed_demo = 5
strat_one = cross_val_score(
    clf_weak, X_dig, y_imb,
    cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed_demo),
    scoring='f1'
)
nonstrat_one = cross_val_score(
    clf_weak, X_dig, y_imb,
    cv=KFold(n_splits=10, shuffle=True, random_state=seed_demo),
    scoring='f1'
)
print(f"\nSingle seed (seed={seed_demo}):")
print(f"  Stratified     F1 std: {strat_one.std():.4f}")
print(f"  Non-stratified F1 std: {nonstrat_one.std():.4f}")

# Why: without stratification the roughly 180 positive digit-8 examples
# land unevenly across the 10 folds, so some folds see far fewer positives
# than others.
print("\nFold-by-fold positive counts (non-stratified, this seed):")
for i, (_, val_idx) in enumerate(
    KFold(n_splits=10, shuffle=True, random_state=seed_demo).split(X_dig)
):
    print(f"  Fold {i+1}: {y_imb[val_idx].sum()} positives out of {len(val_idx)}")

### B3b. Repeating the comparison over many seeds

A single seed is not decisive: fold assignment is random, so either method
can happen to look less variable in one particular split. Repeating the
comparison over many random splits and averaging the variance estimate
(the same logic behind repeated CV, Section 2.3) shows the expected
direction reliably, even though it does not hold for every individual
seed.

In [ ]:
n_seeds = 30
summary = {}
for scoring in ['accuracy', 'precision', 'f1']:
    strat_stds, nonstrat_stds = [], []
    for seed in range(n_seeds):
        s = cross_val_score(
            clf_weak, X_dig, y_imb,
            cv=StratifiedKFold(n_splits=10, shuffle=True, random_state=seed),
            scoring=scoring
        ).std()
        ns = cross_val_score(
            clf_weak, X_dig, y_imb,
            cv=KFold(n_splits=10, shuffle=True, random_state=seed),
            scoring=scoring
        ).std()
        strat_stds.append(s)
        nonstrat_stds.append(ns)
    summary[scoring] = (np.mean(strat_stds), np.mean(nonstrat_stds),
                         sum(n > s for n, s in zip(nonstrat_stds, strat_stds)))

print(f"Averaged over {n_seeds} random seeds:\n")
for scoring, (strat_mean, nonstrat_mean, wins) in summary.items():
    print(f"{scoring:10s}  mean stratified std: {strat_mean:.4f}   "
          f"mean non-stratified std: {nonstrat_mean:.4f}   "
          f"non-stratified higher in {wins}/{n_seeds} seeds")

fig, ax = plt.subplots(figsize=(6, 4))
labels = list(summary.keys())
strat_vals = [summary[k][0] for k in labels]
nonstrat_vals = [summary[k][1] for k in labels]
x_pos = np.arange(len(labels))
width = 0.35
ax.bar(x_pos - width/2, strat_vals, width, label='Stratified', color='steelblue')
ax.bar(x_pos + width/2, nonstrat_vals, width, label='Non-stratified', color='indianred')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel(f'Mean fold-score std over {n_seeds} seeds')
ax.set_title('Stratified vs non-stratified CV variance, by metric')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/b3b_variance_comparison.png', dpi=110, bbox_inches='tight')
plt.close()
print("\nSaved b3b_variance_comparison.png")

**Exercise B.** Why does non-stratified $k$-fold CV produce higher variance in
the fold scores when the data are imbalanced?  Under what conditions would
LOOCV be preferable to 10-fold CV?

## Part C — The multiple comparisons problem and nested CV

When many models are evaluated on the same validation set, the best-performing
model benefits from sampling luck.  Nested CV separates model *selection* (inner
loop) from error *estimation* (outer loop), providing an unbiased estimate of
the error of the entire selection procedure.

### C1. Selection bias with many models

In [ ]:
# Compare many values of max_depth using the same CV split — observe selection bias
rng2 = np.random.default_rng(6)
depth_grid = list(range(1, 21))
n_rep = 20

best_val_acc  = []
true_test_acc = []

X_tr3, X_te3, y_tr3, y_te3 = train_test_split(
    X_dig, y_dig, test_size=0.25, random_state=6, stratify=y_dig
)

for _ in range(n_rep):
    # Each repetition: pick best depth by 5-fold CV on training set
    cv_scores = []
    for d in depth_grid:
        s = cross_val_score(
            DecisionTreeClassifier(max_depth=d),
            X_tr3, y_tr3,
            cv=StratifiedKFold(n_splits=5, shuffle=True,
                               random_state=rng2.integers(1000)),
            scoring='accuracy'
        )
        cv_scores.append(s.mean())
    best_d = depth_grid[np.argmax(cv_scores)]
    best_val_acc.append(max(cv_scores))

    # Evaluate the selected model on the true test set
    clf_best = DecisionTreeClassifier(max_depth=best_d)
    clf_best.fit(X_tr3, y_tr3)
    true_test_acc.append(accuracy_score(y_te3, clf_best.predict(X_te3)))

print(f"Mean selected CV accuracy (optimistic): {np.mean(best_val_acc):.4f}")
print(f"Mean true test accuracy (unbiased):     {np.mean(true_test_acc):.4f}")
print(f"Selection bias:                         {np.mean(best_val_acc)-np.mean(true_test_acc):+.4f}")

### C2. Nested cross-validation

In [ ]:
# Nested CV: outer 5-fold estimates generalisation error of the
# 'fit a decision tree with depth chosen by inner 5-fold CV' procedure.

from sklearn.model_selection import cross_val_score
from sklearn.pipeline import Pipeline

param_grid = {'max_depth': list(range(1, 16))}
inner_cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=7)
outer_cv  = StratifiedKFold(n_splits=5, shuffle=True, random_state=8)

gs = GridSearchCV(
    DecisionTreeClassifier(), param_grid, cv=inner_cv, scoring='accuracy'
)

nested_scores = cross_val_score(gs, X_dig, y_dig, cv=outer_cv, scoring='accuracy')

# Non-nested (uses the same data for selection and evaluation)
gs_nonnested = GridSearchCV(
    DecisionTreeClassifier(), param_grid, cv=inner_cv, scoring='accuracy'
)
gs_nonnested.fit(X_dig, y_dig)
non_nested_score = gs_nonnested.best_score_

print(f"Non-nested CV best score:         {non_nested_score:.4f}")
print(f"Nested CV scores per outer fold:  {nested_scores.round(4)}")
print(f"Nested CV mean:                   {nested_scores.mean():.4f}")
print(f"Optimism (non-nested - nested):   {non_nested_score - nested_scores.mean():+.4f}")

**Exercise C.** In the nested CV above, the inner loop selects
`max_depth` and the outer loop estimates the generalisation error.  Explain
why fitting `GridSearchCV` on all the data and then reporting `best_score_`
is not a valid substitute for the outer loop.

## Part D — Comparing two classifiers: McNemar's test

When two classifiers are evaluated on the same test set, their errors are
correlated (both see the same hard examples).  McNemar's test accounts for
this by looking only at the cases where the two classifiers disagree.

In [ ]:
# Fit two classifiers on training set, collect per-sample correctness on test set
rf_d  = RandomForestClassifier(n_estimators=200, random_state=9)
lr_d  = LogisticRegression(max_iter=500, C=1.0)

rf_d.fit(X_tr, y_tr)
lr_d.fit(X_tr, y_tr)

rf_correct = (rf_d.predict(X_te) == y_te)
lr_correct = (lr_d.predict(X_te) == y_te)

# Contingency table:
#            LR correct   LR wrong
# RF correct     a            b
# RF wrong       c            d
a = np.sum( rf_correct &  lr_correct)
b = np.sum( rf_correct & ~lr_correct)
c = np.sum(~rf_correct &  lr_correct)
d = np.sum(~rf_correct & ~lr_correct)

print(f"Contingency table:")
print(f"              LR correct  LR wrong")
print(f"RF correct       {a:>5}      {b:>5}")
print(f"RF wrong         {c:>5}      {d:>5}")
print()

# McNemar statistic (with continuity correction)
if b + c == 0:
    print("b+c = 0: classifiers agree on all discordant pairs.")
else:
    chi2_stat = (abs(b - c) - 1)**2 / (b + c)
    p_value   = 1 - chi2.cdf(chi2_stat, df=1)
    print(f"McNemar chi2 (with continuity correction) = {chi2_stat:.4f}")
    print(f"p-value = {p_value:.4f}")
    if p_value < 0.05:
        better = "Random Forest" if b > c else "Logistic Regression"
        print(f"Conclusion: {better} is significantly better (p < 0.05).")
    else:
        print("No statistically significant difference at the 5% level.")

In [ ]:
# Visual: which samples does each classifier get right/wrong?
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, correct, name in zip(
    axes,
    [rf_correct, lr_correct],
    ['Random Forest', 'Logistic Regression']
):
    ax.scatter(range(len(correct)), y_te,
               c=['steelblue' if c else 'tomato' for c in correct],
               s=6, alpha=0.6)
    ax.set_title(f'{name}: accuracy={correct.mean():.4f}')
    ax.set_xlabel('Test sample index')
    ax.set_ylabel('True class')

plt.suptitle('Per-sample correctness: blue = correct, red = wrong')
plt.tight_layout()
plt.savefig('/tmp/d1_mcnemar_scatter.png', dpi=110)
plt.close()
print("Saved d1_mcnemar_scatter.png")

**Exercise D.** A naive comparison would simply test whether the two accuracy
values differ using a two-proportion $z$-test.  Explain why this is
inappropriate here, and why McNemar's test is the correct choice.

## Part E — Probability calibration

A well-calibrated classifier outputs probabilities that reflect true frequencies:
if it says 0.7, roughly 70% of those predictions should be correct.
High accuracy does not guarantee calibration.  We measure calibration via
reliability diagrams and the Brier score, then apply Platt scaling.

### E1. Binary calibration on Digits (digit 0 vs rest)

In [ ]:
# Binary problem: is the digit a zero?
y_bin = (y_dig == 0).astype(int)
X_tr_b, X_te_b, y_tr_b, y_te_b = train_test_split(
    X_dig, y_bin, test_size=0.25, random_state=10, stratify=y_bin
)

# Three classifiers with different calibration properties
rf_b  = RandomForestClassifier(n_estimators=200, random_state=10)
lr_b  = LogisticRegression(max_iter=500, C=1.0)
svc_b = SVC(kernel='rbf', probability=True, random_state=10)

for clf, name in [(rf_b, 'Random Forest'),
                  (lr_b, 'Logistic Regression'),
                  (svc_b, 'SVC (Platt)')]:
    clf.fit(X_tr_b, y_tr_b)

probs = {
    'Random Forest':       rf_b.predict_proba(X_te_b)[:, 1],
    'Logistic Regression': lr_b.predict_proba(X_te_b)[:, 1],
    'SVC (Platt)':         svc_b.predict_proba(X_te_b)[:, 1],
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, prob) in zip(axes, probs.items()):
    frac_pos, mean_pred = calibration_curve(y_te_b, prob, n_bins=10)
    brier = brier_score_loss(y_te_b, prob)
    ax.plot(mean_pred, frac_pos, 's-', color='steelblue', label='Calibration curve')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    ax.set_title(f'{name}\nBrier={brier:.4f}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)

plt.suptitle('Reliability diagrams — digit 0 vs rest', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/e1_calibration.png', dpi=110)
plt.close()
print("Saved e1_calibration.png")
for name, prob in probs.items():
    print(f"{name:<25} Brier = {brier_score_loss(y_te_b, prob):.4f}")

### E2. Post-hoc calibration with Platt scaling and isotonic regression

In [ ]:
# Fit Random Forest, then calibrate with sigmoid (Platt) and isotonic regression
rf_uncal = RandomForestClassifier(n_estimators=200, random_state=11)

rf_platt = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=200, random_state=11),
    method='sigmoid', cv=5
)
rf_iso = CalibratedClassifierCV(
    RandomForestClassifier(n_estimators=200, random_state=11),
    method='isotonic', cv=5
)

rf_uncal.fit(X_tr_b, y_tr_b)
rf_platt.fit(X_tr_b, y_tr_b)
rf_iso.fit(X_tr_b, y_tr_b)

models_cal = {
    'Uncalibrated RF':     rf_uncal.predict_proba(X_te_b)[:, 1],
    'Platt scaling':       rf_platt.predict_proba(X_te_b)[:, 1],
    'Isotonic regression': rf_iso.predict_proba(X_te_b)[:, 1],
}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, (name, prob) in zip(axes, models_cal.items()):
    frac_pos, mean_pred = calibration_curve(y_te_b, prob, n_bins=10)
    brier = brier_score_loss(y_te_b, prob)
    ax.plot(mean_pred, frac_pos, 's-', color='steelblue', label='Calibration curve')
    ax.plot([0, 1], [0, 1], 'k--', lw=1, label='Perfect calibration')
    ax.set_title(f'{name}\nBrier={brier:.4f}')
    ax.set_xlabel('Mean predicted probability')
    ax.set_ylabel('Fraction of positives')
    ax.legend(fontsize=8)

plt.suptitle('Calibration before and after post-hoc correction', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/e2_calibration_corrected.png', dpi=110)
plt.close()
print("Saved e2_calibration_corrected.png")
for name, prob in models_cal.items():
    print(f"{name:<25} Brier = {brier_score_loss(y_te_b, prob):.4f}")

**Exercise E.** The Brier score combines calibration and sharpness.  A
classifier that always predicts the base rate achieves a Brier score equal to
$p(1-p)$ where $p$ is the positive class frequency.  Compute this value for the
digit-0 binary problem and compare it to the three models above.  What does
it mean if a model's Brier score exceeds this baseline?

## Part F — SHAP: Explaining Individual Predictions

*Addendum, provisional pending joint review.*

The evaluation tools above tell us how good a model is overall. They say
nothing about why the model made a particular prediction. We revisit the
Random Forest fitted in Part E on the digit-0-vs-rest problem and use
TreeSHAP (Definition 6.3 of the lecture) to attribute each prediction to
individual pixels, then compare the resulting global feature importance
to the model's built-in MDI importance from Week 9.

### F1. TreeSHAP and the additivity property

In [ ]:
import shap

# Reuse the Random Forest fitted on the digit-0-vs-rest problem in Part E1
explainer = shap.TreeExplainer(rf_b)
shap_values_te = explainer.shap_values(X_te_b)
print(f"SHAP value array shape: {shap_values_te.shape}")
print(f"(n_test_instances, n_pixels, n_classes)")

# Efficiency / additivity (Proposition 6.1): the SHAP values for the
# positive class should sum to (predicted probability - expected value)
class_idx = 1
base_value = explainer.expected_value[class_idx]
shap_sum = shap_values_te[:, :, class_idx].sum(axis=1) + base_value
pred_proba = rf_b.predict_proba(X_te_b)[:, class_idx]

print(f"\nExpected value (base rate), class 1: {base_value:.4f}")
print(f"Max absolute difference, sum(SHAP)+base vs predict_proba: "
      f"{np.max(np.abs(shap_sum - pred_proba)):.2e}")
print("Additivity confirmed:", np.allclose(shap_sum, pred_proba, atol=1e-6))

### F2. Global importance: mean |SHAP| versus MDI

Averaging the absolute SHAP value of each pixel across the test set gives
a model-level importance ranking, directly comparable to the MDI (Gini)
importance from Week 9's Random Forest. Both are visualised as 8x8 pixel
heatmaps, since each Digits feature corresponds to one pixel position.

In [ ]:
mean_abs_shap = np.abs(shap_values_te[:, :, 1]).mean(axis=0)
mdi_importance = rf_b.feature_importances_

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
im0 = axes[0].imshow(mean_abs_shap.reshape(8, 8), cmap='viridis')
axes[0].set_title('Mean |SHAP value| per pixel')
axes[0].axis('off')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(mdi_importance.reshape(8, 8), cmap='viridis')
axes[1].set_title('MDI (Gini) importance per pixel')
axes[1].axis('off')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

plt.suptitle('Global feature importance: SHAP vs MDI (digit 0 vs rest)', y=1.02)
plt.tight_layout()
plt.savefig('/tmp/f2_shap_vs_mdi.png', dpi=110, bbox_inches='tight')
plt.close()
print('Saved f2_shap_vs_mdi.png')

rank_corr = np.corrcoef(mean_abs_shap, mdi_importance)[0, 1]
print(f"Pearson correlation between the two importance rankings: {rank_corr:.3f}")

### F3. Local explanations: a correct and a misclassified instance

A local explanation attributes one specific prediction, rather than an
average over the dataset. We contrast a test instance the model gets
right against one it gets wrong, plotting the pixel-level SHAP values as
a diverging heatmap next to the actual digit image: red pixels pushed the
prediction toward "digit 0", blue pixels pushed it away.

In [ ]:
pred_te = rf_b.predict(X_te_b)
correct_idx = np.where((pred_te == y_te_b) & (y_te_b == 1))[0][0]
wrong_idx_candidates = np.where(pred_te != y_te_b)[0]
wrong_idx = wrong_idx_candidates[0] if len(wrong_idx_candidates) > 0 else correct_idx

fig, axes = plt.subplots(2, 2, figsize=(7, 7))
for row, idx in enumerate([correct_idx, wrong_idx]):
    image = X_te_b[idx].reshape(8, 8)
    shap_map = shap_values_te[idx, :, 1].reshape(8, 8)
    vmax = np.abs(shap_map).max()

    axes[row, 0].imshow(image, cmap='gray')
    axes[row, 0].set_title(f"True: {'digit 0' if y_te_b[idx]==1 else 'other'}, "
                           f"predicted: {'digit 0' if pred_te[idx]==1 else 'other'}")
    axes[row, 0].axis('off')

    im = axes[row, 1].imshow(shap_map, cmap='coolwarm', vmin=-vmax, vmax=vmax)
    axes[row, 1].set_title('SHAP attribution (class: digit 0)')
    axes[row, 1].axis('off')
    plt.colorbar(im, ax=axes[row, 1], fraction=0.046)

plt.suptitle('Local SHAP explanations: correctly classified (top) vs '
             'misclassified (bottom)', y=1.01)
plt.tight_layout()
plt.savefig('/tmp/f3_local_explanations.png', dpi=110, bbox_inches='tight')
plt.close()
print('Saved f3_local_explanations.png')
print(f"Correctly classified instance: index {correct_idx}")
print(f"Misclassified instance:        index {wrong_idx}")

**Exercise F.** For the misclassified instance plotted in F3, identify the pixels with the largest-magnitude SHAP values and describe, in terms of the digit's shape, why those pixels might have misled the classifier. The mean |SHAP| and MDI importance maps in F2 need not agree in every pixel; suggest one reason the two rankings could differ, referring to the consistency property discussed in the lecture (Remark 6.5).

---

## Take-home exercises

**Exercise 1 — The one-standard-error rule.**
Using the Digits dataset and a Random Forest with `n_estimators=100`, perform
10-fold stratified CV over `max_depth` $\in \{2, 3, 5, 8, 12, \text{None}\}$.
Plot the mean CV accuracy with error bars of one standard error across folds.
Apply the one-standard-error rule: select the simplest model whose mean accuracy
is within one standard error of the best mean accuracy.  Does it differ from the
naive argmax?

**Exercise 2 — Nested CV on Wine.**
Using the Wine dataset, set up nested CV ($5 \times 5$) where the inner loop
selects the best $C \in \{0.01, 0.1, 1, 10, 100\}$ for a logistic regression,
and the outer loop estimates generalisation error.  Report the non-nested
best score and the nested mean score, and explain any difference.

**Exercise 3 — McNemar's test on Wine.**
Train a Random Forest ($B=200$) and a Gradient Boosting classifier (200 stages,
`max_depth=3`) on the Wine dataset.  Evaluate both on a held-out test set and
apply McNemar's test.  Is the difference statistically significant?

**Exercise 4 — Calibration of a multiclass classifier.**
The `calibration_curve` function works only for binary problems.  For the
full 10-class Digits dataset, compute the Brier score for a Random Forest by
using one-vs-rest binarisation: for each class $c$, binarise the labels and
compute the Brier score using `predict_proba(X_te)[:, c]`.  Average the
per-class Brier scores.  Apply Platt scaling and check whether the average
Brier score improves.